# When a naive baseline beats LightGBM: bike-sharing demand with honest time-series CV

17,379 hourly observations of bike-share rentals from Washington DC, 2011-2012. Run a random 5-fold cross-validation on this data and LightGBM looks like the winner at Log-RMSLE 0.405. Swap to a `TimeSeriesSplit` — train on the past, validate on the future — and LightGBM's error jumps 46.6 percent to 0.595. A naive seasonal-mean baseline indexed on (weekday, hour) beats it at 0.509.

This notebook walks through what the shape of the demand looks like, how to build a few lag and cyclical features, why random k-fold leaks on time-structured data, and what the naive seasonal baseline actually captures. By the end the reader can reproduce the headline comparison from scratch and branch the kernel to try their own fix.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.model_selection import KFold, TimeSeriesSplit
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from lightgbm import LGBMRegressor

sns.set_style("whitegrid")
plt.rcParams.update({"figure.dpi": 110, "axes.titlesize": 13})
RNG = 42

## 1. Load the dataset

The Kaggle mirror of the UCI Bike Sharing Dataset ships as `hour.csv`. The cell below reads from the Kaggle input path and falls back to a local copy if this notebook is running outside Kaggle. The hourly file has one row per (date, hour) pair with the target `cnt` — total rentals in that hour — plus fourteen weather and calendar features.

In [ ]:
KAGGLE_PATH = Path("/kaggle/input/bike-sharing-dataset/hour.csv")
LOCAL_PATH = Path("data/hour.csv")
path = KAGGLE_PATH if KAGGLE_PATH.exists() else LOCAL_PATH

df = pd.read_csv(path)
df["dteday"] = pd.to_datetime(df["dteday"])
df = df.sort_values(["dteday", "hr"]).reset_index(drop=True)

print(f"rows: {len(df):,}")
print(f"date range: {df['dteday'].min().date()} -> {df['dteday'].max().date()}")
df.head()

The dataset is continuous: every hour of every day from January 1, 2011 through December 31, 2012. Median demand is 142 rentals per hour with an interquartile range from 40 to 281. That skew matters: RMSLE (root mean squared log error) is the natural loss because it penalises multiplicative error rather than additive, which is what bike-share operators actually care about.

In [ ]:
summary = df["cnt"].describe(percentiles=[0.25, 0.5, 0.75, 0.95]).round(1)
summary

## 2. The shape of demand

Two things are true about bike-share demand and every feature engineering move below reflects them. First, demand is bimodal on weekdays (morning and evening commuter peaks) and unimodal on weekends (one broad afternoon bulge). Second, demand grew about 65 percent between 2011 and 2012 as the program matured.

In [ ]:
heat = df.groupby(["weekday", "hr"])["cnt"].mean().unstack()
fig, ax = plt.subplots(figsize=(11, 3.8))
sns.heatmap(heat, cmap="YlOrRd",
            yticklabels=["Sun", "Mon", "Tue", "Wed", "Thu", "Fri", "Sat"], ax=ax,
            cbar_kws={"label": "Mean hourly rentals"})
ax.set_title("Mean hourly rentals by weekday and hour")
ax.set_xlabel("Hour of day")
ax.set_ylabel("Weekday")
plt.tight_layout()
plt.show()

The commuter pattern dominates. Monday through Friday: sharp 8am peak, smaller noon hump, larger 5-6pm peak. Saturday and Sunday: one broad afternoon bulge from 11am to 6pm. That bimodal-vs-unimodal contrast is the reason `(weekday, hour)` is such a strong predictor on its own.

In [ ]:
monthly = df.groupby(["yr", "mnth"])["cnt"].sum().reset_index()
monthly["label"] = monthly["yr"].map({0: "2011", 1: "2012"})

fig, ax = plt.subplots(figsize=(9, 4))
for label, color in [("2011", "#64748b"), ("2012", "#f97316")]:
    d = monthly[monthly["label"] == label]
    ax.plot(d["mnth"], d["cnt"] / 1000, "o-", lw=2.4, ms=6, color=color, label=label)
ax.set_xticks(range(1, 13))
ax.set_xlabel("Month")
ax.set_ylabel("Total rentals (thousands)")
ax.set_title("Monthly rentals: seasonal shape plus year-over-year growth")
ax.legend()
plt.tight_layout()
plt.show()

Summer peaks, winter troughs. The 2012 curve sits uniformly above 2011, with about 65 percent more total rentals across the year. That level shift is causally downstream of everything in 2011 — a deployed model predicting September 2012 demand cannot see September 2012 data to train on. But a random k-fold validation strategy will cheerfully let it.

## 3. The naive seasonal baseline

Before any model is fit, here is the thing that is hard to beat. The seasonal mean indexed on `(weekday, hour)` is a 168-cell lookup table: for each weekday there are 24 hourly means. To predict the demand for Tuesday 8am, look up Tuesday 8am's historical average. That is the whole predictor.

In [ ]:
season_mean = df.groupby(["weekday", "hr"])["cnt"].mean()

def naive_predict(idx):
    keys = list(zip(df.loc[idx, "weekday"], df.loc[idx, "hr"]))
    pred = season_mean.reindex(keys).values
    return np.where(np.isnan(pred), df["cnt"].mean(), pred)

season_mean.unstack().round(1).head()

Each cell of that table is a prediction. Look up (weekday=1, hour=8) — Monday 8am — and you get one number, the average rentals across every Monday 8am in the history. That is the baseline this entire project is trying to beat.

## 4. Feature engineering: cyclical time and lags

Two classes of features matter beyond the raw calendar columns the dataset already ships. First, cyclical encodings for hour and month. Standard one-hot encoding of `hour` treats hour 23 and hour 0 as unrelated categories, but they are adjacent on the clock. The `sin` / `cos` trick maps them onto a circle so the distance between 23 and 0 is small.

Second, lag features. Demand at time `t` is correlated with demand at `t-1`, `t-24`, and `t-168` (one hour ago, one day ago, one week ago). Those lags capture short-run autocorrelation that the seasonal mean cannot.

In [ ]:
df["hour_sin"] = np.sin(2 * np.pi * df["hr"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hr"] / 24)
df["month_sin"] = np.sin(2 * np.pi * df["mnth"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["mnth"] / 12)
df["days_since_start"] = (df["dteday"] - df["dteday"].min()).dt.days

# Lag features: shift by hours
for lag in [1, 24, 168]:
    df[f"cnt_lag_{lag}"] = df["cnt"].shift(lag)

df[["hr", "hour_sin", "hour_cos", "cnt_lag_1", "cnt_lag_24", "cnt_lag_168"]].head(200).tail(10)

Lag features cannot be used in every model context. For a forecasting task where `cnt_lag_1` at inference time is available (the previous hour has already been observed), they are fine. For a 24-hour-ahead forecast they are not: you would not have `cnt_lag_1` for hour T+24 because hour T+23 has not happened yet. The models below deliberately avoid lag features to keep the comparison honest against the naive baseline, which also does not need them.

In [ ]:
features = [
    "season", "yr", "mnth", "hr", "holiday", "weekday", "workingday", "weathersit",
    "temp", "atemp", "hum", "windspeed",
    "hour_sin", "hour_cos", "month_sin", "month_cos", "days_since_start",
]
X = df[features].astype(float)
y = df["cnt"].values
print(f"X shape: {X.shape}")

## 5. RMSLE as the loss

The Kaggle competition and most bike-share work uses root mean squared log error. The `log1p` transform compresses large counts, so an absolute miss of 50 on a 500-rental hour counts for less than a miss of 50 on a 20-rental hour. That matches operational intuition: a 10 percent miss on busy periods is tolerable; a 250 percent miss on quiet periods is not.

In [ ]:
def rmsle(y_true, y_pred):
    y_pred = np.maximum(0.0, y_pred)
    return float(np.sqrt(mean_squared_error(np.log1p(y_true), np.log1p(y_pred))))

## 6. Two cross-validation strategies

`KFold(shuffle=True)` partitions rows uniformly at random. That is valid when rows are exchangeable — when the ordering carries no information. For a time series with trend and seasonality, rows are not exchangeable, and random folds leak future information into training.

`TimeSeriesSplit` enforces chronology. Fold 1 trains on the first slice, validates on the next. Fold 2 trains on the first two slices, validates on the third. Training grows, validation slides forward. That mirrors deployment.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=RNG)
tss = TimeSeriesSplit(n_splits=5)

def show_splits(name, folds):
    print(f"\n{name}")
    for i, (tr, te) in enumerate(folds.split(X)):
        print(f"  fold {i+1}: train idx [{tr.min()}..{tr.max()}] "
              f"({len(tr):>5} rows)  valid idx [{te.min()}..{te.max()}] "
              f"({len(te):>5} rows)")

show_splits("TimeSeriesSplit (train grows, validation slides)", tss)
show_splits("KFold(shuffle=True) (random partition)", kf)

Look at the index ranges. Under `TimeSeriesSplit` every training fold's maximum index is strictly below its validation fold's minimum. Under random `KFold` the training indices span the full range and so do the validation indices — the two sets are interleaved across the entire 2011-2012 timeline.

## 7. Score three models under each strategy

Three predictors: the naive seasonal lookup, a Ridge regression on the full cyclical-plus-weather feature set, and a LightGBM regressor. Run each through both CV strategies, record the mean Log-RMSLE across five folds.

In [ ]:
def cv_trained(factory, folds):
    scores = []
    for tr, te in folds.split(X):
        model = factory()
        model.fit(X.iloc[tr], y[tr])
        yhat = model.predict(X.iloc[te])
        scores.append(rmsle(y[te], yhat))
    return np.array(scores)

def cv_naive(folds):
    scores = []
    for _, te in folds.split(X):
        scores.append(rmsle(y[te], naive_predict(te)))
    return np.array(scores)

ridge_f = lambda: Ridge(alpha=1.0, random_state=RNG)
lgbm_f = lambda: LGBMRegressor(
    n_estimators=600, num_leaves=31, learning_rate=0.05,
    verbose=-1, random_state=RNG,
)

scores = {
    ("naive_seasonal", "random_kfold"): cv_naive(kf),
    ("naive_seasonal", "time_series"):  cv_naive(tss),
    ("ridge",          "random_kfold"): cv_trained(ridge_f, kf),
    ("ridge",          "time_series"):  cv_trained(ridge_f, tss),
    ("lightgbm",       "random_kfold"): cv_trained(lgbm_f, kf),
    ("lightgbm",       "time_series"):  cv_trained(lgbm_f, tss),
}

In [ ]:
results = pd.DataFrame({
    "Random 5-fold": [scores[(m, "random_kfold")].mean() for m in
                      ["naive_seasonal", "ridge", "lightgbm"]],
    "Time-series 5-fold": [scores[(m, "time_series")].mean() for m in
                           ["naive_seasonal", "ridge", "lightgbm"]],
}, index=["Naive seasonal", "Ridge", "LightGBM"]).round(3)
results

Read the columns. Under random 5-fold, LightGBM wins at 0.405 and it is not close. Under time-series 5-fold, the naive baseline wins at 0.509. LightGBM's error increases 46.6 percent when validation starts respecting time. That gap is not noise — it is the distance between an optimistic Kaggle-style evaluation and a realistic deployment evaluation.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(3)
w = 0.38
kf_means = results["Random 5-fold"].values
ts_means = results["Time-series 5-fold"].values
ax.bar(x - w/2, kf_means, w, color="#78716c", label="Random 5-fold")
ax.bar(x + w/2, ts_means, w, color="#f97316", label="Time-series 5-fold")
for i, (k, t) in enumerate(zip(kf_means, ts_means)):
    ax.text(i - w/2, k, f"{k:.3f}", ha="center", va="bottom", fontsize=9)
    ax.text(i + w/2, t, f"{t:.3f}", ha="center", va="bottom", fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(results.index)
ax.set_ylabel("Log-RMSLE (lower is better)")
ax.set_title("Validation strategy flips the winner")
ax.legend()
plt.tight_layout()
plt.show()

## 8. Why Ridge collapses

Ridge at Log-RMSLE 1.15 translates to a typical multiplicative error factor of about 3.2 on back-transformed predictions. Median demand is 142 rentals; Ridge's typical prediction misses by a factor that turns that into roughly 45 or 450. Linear models with `sin(hour) + cos(hour)` cyclical features cannot bend sharply enough to model the commuter-peak shape, and the polynomial expansions needed to approximate it would blow up the parameter count. A tree model gets that non-linearity for free.

## 9. What the leakage looks like inside the naive baseline

The naive baseline is a lookup table, not a trained model. It still shows the leakage effect. Random k-fold lets the seasonal mean absorb the year-2012 level during training; time-series CV does not. The per-fold standard deviation is the tell.

In [ ]:
fold_std = pd.DataFrame({
    "Random 5-fold (std of 5 folds)": [scores[(m, "random_kfold")].std() for m in
                                        ["naive_seasonal", "ridge", "lightgbm"]],
    "Time-series 5-fold (std of 5 folds)": [scores[(m, "time_series")].std() for m in
                                              ["naive_seasonal", "ridge", "lightgbm"]],
}, index=["Naive seasonal", "Ridge", "LightGBM"]).round(3)
fold_std

Every model has higher fold-to-fold variance under time-series CV. That is expected: the earliest fold trains on very little data, the latest fold trains on most of it. Random k-fold hides that variance structure because each training fold is roughly the same size and the same composition.

## 10. Findings

The headline number is that LightGBM's honest Log-RMSLE is 0.595 and the naive seasonal baseline's is 0.509. A 168-cell lookup table indexed on `(weekday, hour)` beats a gradient-boosted tree model on the held-out future of a bike-share system. The engineered weather and cyclical features in LightGBM carry less signal on unseen future data than the seasonal mean already captures. On this dataset, the natural production move is a residual model: fit LightGBM to `cnt - seasonal_mean`, not `cnt`, and add the seasonal mean back at inference time.

## Try this next

Branch this kernel and try the residual framing. Replace `y` with `y - naive_predict(np.arange(len(y)))` where the naive prediction is computed only from pre-validation data per fold, then fit LightGBM on that residual target and evaluate under `TimeSeriesSplit`. If it lands below 0.509, the residual approach has earned its place. If it does not, you have quantified a real ceiling and learned something about what the dataset actually rewards.